In [1]:
from email import utils
import polars as pl
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon, Point
import matplotlib.pyplot as plt
from pathlib import Path
import sys
sys.path.append("../../src")  # Add path to where 'retail' package lives
import retail.retailfilter as rf # Correct import path
import retail.retailfilter_new as rfn
import utils.utils as ut  # Correct import path


ModuleNotFoundError: No module named 'retail'

In [ ]:

# Input file
input_file = Path("../../data/input_files/uk_foursquare_datefields_admin_categories.parquet")

# Read in parquet
foursquare = pl.read_parquet(input_file)

# Filter
foursquare_filtnew = rfn.filter_foursquare(foursquare)
print(len(foursquare_filtnew))

# Columns to keep
COLUMNS_TO_KEEP = [
    "fsq_place_id", "name", "lat", "lng", "address", "locality", "region",
    "postcode", "admin_region", "post_town", "po_box", "country",
    "date_created", "date_refreshed", "date_closed", "tel", "website", 
    "fsq_category_ids", "fsq_category_labels", "h3_11"
]

# Add H3 column
foursquare_filtnew = ut.add_h3_11_column(foursquare_filtnew)

# Keep only selected columns
foursquare_filtnew = foursquare_filtnew.select(COLUMNS_TO_KEEP)

# Convert to pandas
fsq_df = foursquare_filtnew.to_pandas()

# Create geometry
fsq_df["geometry"] = [
    Point(xy) if pd.notnull(xy[0]) and pd.notnull(xy[1]) else None
    for xy in zip(fsq_df["lng"], fsq_df["lat"])
]
fsq_df = fsq_df[fsq_df["geometry"].notnull()]

# Convert to GeoDataFrame
gdf = gpd.GeoDataFrame(fsq_df, geometry="geometry", crs="EPSG:4326")

# Generate output filename: "inputfilename_retailfiltnew_geo.parquet"
output_path = input_file.with_name(input_file.stem + "_retailfilt_geo.parquet")

# Save
gdf.to_parquet(output_path)
print(f"✅ Saved GeoDataFrame with H3 to {output_path}")

# Generate output filename: GeoPackage
output_gpkg = input_file.with_name(input_file.stem + "_retailfilt_geo.gpkg")
gdf.to_file(output_gpkg, driver="GPKG")
print(f"✅ Also saved GeoDataFrame as GeoPackage to {output_gpkg}")

Filtered Foursquare data: 1910720 rows remaining.
1560149
✅ Saved GeoDataFrame with H3 to ../../data/input_files/uk_foursquare_datefields_admin_categories_retailfiltnew_geo.parquet
